Load Data

In [73]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

Settings for Pandas and visualization

In [74]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)
sns.set_style('whitegrid')
sns.set_palette('viridis')

Loading CC_Calls cleaned data

In [75]:
df_cc_calls = pd.read_csv("../../../data/interim/cc_calls_cleaned.csv", )
df_billings = pd.read_csv('../../../data/interim/billings_features.csv')

df_renewal_dates = df_billings[['co_ref', 'renewal_month']].copy()

print(f"CC calls shape : {df_cc_calls.shape}")
print(f"Unique customers    : {df_cc_calls['co_ref'].nunique()}")

CC calls shape : (31636, 30)
Unique customers    : 14988


Parse Dates

In [76]:
df_cc_calls['call_date'] = pd.to_datetime(df_cc_calls['call_date'], errors='coerce')

df_renewal_dates['prospect_renewal_date'] = pd.to_datetime(
    df_renewal_dates['renewal_month'], errors='coerce'
)

Creating Feature for Cutoff Date

In [77]:
df_renewal_dates['cutoff_date'] = df_renewal_dates['prospect_renewal_date'] - pd.Timedelta(days=14)

Merge CC_calls Date onto calls

In [78]:
df_cc_calls = df_cc_calls.merge(
    df_renewal_dates,
    on='co_ref',
    how='inner'
)

print(f"Calls with cc_calls date matched: {df_cc_calls['prospect_renewal_date'].notna().sum()}")
print(f"Calls with no match            : {df_cc_calls['prospect_renewal_date'].isna().sum()}")

Calls with cc_calls date matched: 88830
Calls with no match            : 0


Apply 14 Days filter

In [79]:
df_cc_calls = df_cc_calls[
    df_cc_calls['call_date'] <= df_cc_calls['cutoff_date']
]

df_cc_calls_14 = df_cc_calls.copy()

print(f"Total calls              : {len(df_cc_calls):,}")
print(f"Calls in 14-day window   : {len(df_cc_calls_14):,}")
print(f"Calls outside window     : {len(df_cc_calls) - len(df_cc_calls_14):,}")
print(f"Customers with 14d calls : {df_cc_calls_14['co_ref'].nunique():,}")

Total calls              : 9,086
Calls in 14-day window   : 9,086
Calls outside window     : 0
Customers with 14d calls : 5,003


Convert Yes/No/Unknown categorical columns to binary

In [80]:
# Yes = 1, No = 0, Unknown = 0 (no confirmed signal)

yes_no_cols = [
    'cc_care_package_discussed',
    'cc_urgency_getting_on_site',
    'cc_external_consultant',
    'cc_agent_cross_sell_attempt',
    'cc_customer_issues_concerns',
    'cc_business_struggles_financial_hardship',
    'cc_questionnaire_completion',
    'cc_chasing_response',
    'cc_issues_within_questionnaire',
    'cc_login_issues',
    'cc_platform_issues',
    'cc_dissatisfaction_time_to_complete',
    'cc_process_complexity_concerns',
    'cc_questions_harder_than_expected',
    'cc_dissatisfaction_support',
    'cc_pricing_mentioned',
    'cc_pricing_sentiment_impact',
    'cc_refund_discussed',
    'cc_contractor_suggest_leave',
    'cc_contractor_complained',
]

for col in yes_no_cols:
    df_cc_calls_14[col] = df_cc_calls_14[col].map({'Yes': 1, 'No': 0, 'Unknown': 0})

Encoding Sentiment Column

In [81]:
# Dissatisfied = -1, Neutral = 0, Satisfied = 1
sentiment_map = {
    'Satisfied'    :  1,
    'Neutral'      :  0,
    'Not Discussed':  0,
    'Dissatisfied' : -1,
    'Unknown'      :  0,
}
 
df_cc_calls_14['cc_sentiment_encoded'] = df_cc_calls_14['cc_contractor_sentiment'].map(sentiment_map)
df_cc_calls_14['cc_dissatisfied_flag'] = (df_cc_calls_14['cc_contractor_sentiment'] == 'Dissatisfied').astype(int)

Encoding Direction Column

In [82]:
# IN_BOUND = 1, OUT_BOUND = 0
df_cc_calls_14['is_inbound'] = (df_cc_calls_14['direction'] == 'IN_BOUND').astype(int)

One hot encoding

In [83]:
ohe_cols = ['cc_care_package', 'cc_call_initiated_by']

for col in ohe_cols:
    df_cc_calls_14[col] = (
        df_cc_calls_14[col]
        .str.lower()
        .str.replace(' ', '_')
    )
    
df_cc_calls_14 = pd.get_dummies(
    df_cc_calls_14,
    columns=ohe_cols,
    drop_first=False,
    dtype=int
)

Feature engineering


In [84]:
# Sentiment improved or worsened during the call
df_cc_calls_14['sentiment_improved'] = (
    df_cc_calls_14['cc_contractor_sentiment_end_score'] > 
    df_cc_calls_14['cc_contractor_sentiment_start_score']
).astype(int)
 
df_cc_calls_14['sentiment_worsened'] = (
    df_cc_calls_14['cc_contractor_sentiment_end_score'] < 
    df_cc_calls_14['cc_contractor_sentiment_start_score']
).astype(int)
 
# Sentiment change score
df_cc_calls_14['sentiment_change'] = (
    df_cc_calls_14['cc_contractor_sentiment_end_score'] - 
    df_cc_calls_14['cc_contractor_sentiment_start_score']
)
 
# High risk call — multiple issues in one call
df_cc_calls_14['cc_high_risk_call'] = (
    (df_cc_calls_14['cc_contractor_complained']          == 1) |
    (df_cc_calls_14['cc_contractor_suggest_leave']        == 1) |
    (df_cc_calls_14['cc_business_struggles_financial_hardship'] == 1) |
    (df_cc_calls_14['cc_refund_discussed']               == 1)
).astype(int)
 
# Dissatisfaction issue count per call
dissatisfaction_cols = [
    'cc_dissatisfaction_time_to_complete',
    'cc_process_complexity_concerns',
    'cc_questions_harder_than_expected',
    'cc_dissatisfaction_support',
    'cc_platform_issues',
    'cc_login_issues',
]
df_cc_calls_14['cc_dissatisfaction_issue_count'] = df_cc_calls_14[dissatisfaction_cols].sum(axis=1)
 
# Pricing pressure flag — pricing mentioned and had sentiment impact
df_cc_calls_14['pricing_pressure_flag'] = (
    (df_cc_calls_14['cc_pricing_mentioned']        == 1) &
    (df_cc_calls_14['cc_pricing_sentiment_impact'] == 1)
).astype(int)

Aggregate Per Customer using co_ref

In [85]:
agg_dict = {
    # Call counts
    'contact_id'                                    : 'count',
    'is_inbound'                                    : 'sum',

    # Sentiment
    'cc_sentiment_encoded'                          : 'mean',
    'cc_dissatisfied_flag'                          : 'max',
    'cc_contractor_sentiment_start_score'           : 'mean',
    'cc_contractor_sentiment_end_score'             : 'mean',
    'cc_contractor_sentiment_overall_score'         : 'mean',

    # Issue flags
    'cc_customer_issues_concerns'                   : 'max',
    'cc_business_struggles_financial_hardship'      : 'max',
    'cc_login_issues'                               : 'max',
    'cc_platform_issues'                            : 'max',
    'cc_dissatisfaction_time_to_complete'           : 'max',
    'cc_process_complexity_concerns'                : 'max',
    'cc_questions_harder_than_expected'             : 'max',
    'cc_dissatisfaction_support'                    : 'max',
    'cc_pricing_mentioned'                          : 'max',
    'cc_pricing_sentiment_impact'                   : 'max',
    'cc_refund_discussed'                           : 'max',
    'cc_contractor_suggest_leave'                   : 'max',
    'cc_contractor_complained'                      : 'max',
    'cc_urgency_getting_on_site'                    : 'max',
    'cc_chasing_response'                           : 'max',
    'cc_questionnaire_completion'                   : 'max',
    'cc_care_package_discussed'                     : 'max',
    'cc_external_consultant'                        : 'max',
    'cc_agent_cross_sell_attempt'                   : 'max',
    'cc_issues_within_questionnaire'                : 'max',

    # Engineered features
    'sentiment_improved'                            : 'max',
    'sentiment_worsened'                            : 'max',
    'sentiment_change'                              : 'mean',
    'cc_high_risk_call'                                : 'max',
    'cc_dissatisfaction_issue_count'                   : 'sum',
    'pricing_pressure_flag'                         : 'max',

    # OHE — care package type
    'cc_care_package_assisted'                      : 'max',
    'cc_care_package_express'                       : 'max',
    'cc_care_package_not_discussed'                 : 'max',
    'cc_care_package_premier'                       : 'max',
    'cc_care_package_standard'                      : 'max',
    'cc_care_package_unknown'                       : 'max',

    # OHE — call initiated by
    'cc_call_initiated_by_agent'                    : 'sum',
    'cc_call_initiated_by_customer'                 : 'sum',
    'cc_call_initiated_by_not_relevant'             : 'sum',
    'cc_call_initiated_by_unknown'                  : 'sum',
}

df_cc_calls_agg = df_cc_calls_14.groupby('co_ref').agg(agg_dict).reset_index()

df_cc_calls_agg.rename(columns={
    'contact_id' : 'cc_calls_14d_total_calls',
    'is_inbound' : 'cc_calls_14d_inbound_calls',
}, inplace=True)

 
print(f"Shape after aggregation: {df_cc_calls_agg.shape}")
df_cc_calls_agg.head()

Shape after aggregation: (5003, 44)


,co_ref,cc_calls_14d_total_calls,cc_calls_14d_inbound_calls,cc_sentiment_encoded,cc_dissatisfied_flag,cc_contractor_sentiment_start_score,cc_contractor_sentiment_end_score,cc_contractor_sentiment_overall_score,cc_customer_issues_concerns,cc_business_struggles_financial_hardship,cc_login_issues,cc_platform_issues,cc_dissatisfaction_time_to_complete,cc_process_complexity_concerns,cc_questions_harder_than_expected,cc_dissatisfaction_support,cc_pricing_mentioned,cc_pricing_sentiment_impact,cc_refund_discussed,cc_contractor_suggest_leave,cc_contractor_complained,cc_urgency_getting_on_site,cc_chasing_response,cc_questionnaire_completion,cc_care_package_discussed,cc_external_consultant,cc_agent_cross_sell_attempt,cc_issues_within_questionnaire,sentiment_improved,sentiment_worsened,sentiment_change,cc_high_risk_call,cc_dissatisfaction_issue_count,pricing_pressure_flag,cc_care_package_assisted,cc_care_package_express,cc_care_package_not_discussed,cc_care_package_premier,cc_care_package_standard,cc_care_package_unknown,cc_call_initiated_by_agent,cc_call_initiated_by_customer,cc_call_initiated_by_not_relevant,cc_call_initiated_by_unknown
0,AA0784,1,0,1.0,0,50.0,80.0,80.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,30.0,0,0,0,0,0,1,0,0,0,0,1,0,0
1,AA0923,1,1,1.0,0,50.0,90.0,90.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,40.0,0,0,0,0,0,1,0,0,0,0,1,0,0
2,AA0994,2,2,1.0,0,65.0,90.0,87.5,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,1,0,25.0,0,1,0,0,0,1,0,0,0,0,2,0,0
3,AA1155,1,0,1.0,0,80.0,90.0,85.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,10.0,0,0,0,0,0,0,0,1,0,1,0,0,0
4,AA1392,1,0,0.0,0,50.0,80.0,80.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,30.0,0,0,0,0,0,1,0,0,0,1,0,0,0


Add Outbound Calls Column

In [86]:
df_cc_calls_agg['cc_calls_14d_outbound_calls'] = (
    df_cc_calls_agg['cc_calls_14d_total_calls'] - df_cc_calls_agg['cc_calls_14d_inbound_calls']
)

Final check for nulls and shape

In [87]:
print("Nulls:")
print(df_cc_calls_agg.isnull().sum()[df_cc_calls_agg.isnull().sum() > 0])
 
print(f"\nShape: {df_cc_calls_agg.shape}")
print(f"Unique customers: {df_cc_calls_agg['co_ref'].nunique()}")
df_cc_calls_agg.describe()

Nulls:
Series([], dtype: int64)

Shape: (5003, 45)
Unique customers: 5003


,cc_calls_14d_total_calls,cc_calls_14d_inbound_calls,cc_sentiment_encoded,cc_dissatisfied_flag,cc_contractor_sentiment_start_score,cc_contractor_sentiment_end_score,cc_contractor_sentiment_overall_score,cc_customer_issues_concerns,cc_business_struggles_financial_hardship,cc_login_issues,cc_platform_issues,cc_dissatisfaction_time_to_complete,cc_process_complexity_concerns,cc_questions_harder_than_expected,cc_dissatisfaction_support,cc_pricing_mentioned,cc_pricing_sentiment_impact,cc_refund_discussed,cc_contractor_suggest_leave,cc_contractor_complained,cc_urgency_getting_on_site,cc_chasing_response,cc_questionnaire_completion,cc_care_package_discussed,cc_external_consultant,cc_agent_cross_sell_attempt,cc_issues_within_questionnaire,sentiment_improved,sentiment_worsened,sentiment_change,cc_high_risk_call,cc_dissatisfaction_issue_count,pricing_pressure_flag,cc_care_package_assisted,cc_care_package_express,cc_care_package_not_discussed,cc_care_package_premier,cc_care_package_standard,cc_care_package_unknown,cc_call_initiated_by_agent,cc_call_initiated_by_customer,cc_call_initiated_by_not_relevant,cc_call_initiated_by_unknown,cc_calls_14d_outbound_calls
count,5003.000000,5003.000000,5003.000000,5003.000000,5003.000000,5003.000000,5003.000000,5003.000000,5003.000000,5003.000000,5003.000000,5003.000000,5003.000000,5003.000000,5003.000000,5003.000000,5003.000000,5003.000000,5003.000000,5003.000000,5003.000000,5003.000000,5003.000000,5003.000000,5003.000000,5003.000000,5003.000000,5003.000000,5003.000000,5003.000000,5003.000000,5003.000000,5003.000000,5003.000000,5003.000000,5003.000000,5003.000000,5003.000000,5003.000000,5003.000000,5003.000000,5003.000000,5003.000000,5003.000000
mean,1.816110,0.417350,0.483957,0.046772,52.662218,79.082950,81.603134,0.153508,0.044973,0.078153,0.115131,0.070358,0.138717,0.007196,0.032181,0.184090,0.049170,0.009194,0.031781,0.103738,0.142315,0.296022,0.279632,0.302618,0.143314,0.067160,0.186488,0.891265,0.014191,26.420732,0.144713,0.540476,0.048571,0.009994,0.105137,0.832101,0.026384,0.160704,0.010794,0.470917,1.227064,0.107136,0.010993,1.398761
std,1.493343,0.826463,0.509808,0.211171,11.308761,14.821277,10.738017,0.360512,0.207266,0.268439,0.319212,0.255775,0.345685,0.084530,0.176497,0.387596,0.216245,0.095456,0.175434,0.304951,0.349407,0.456547,0.448863,0.459437,0.350428,0.250323,0.389539,0.311337,0.118292,15.130089,0.351847,1.203466,0.214991,0.099479,0.306760,0.373814,0.160291,0.367294,0.103340,0.768809,1.337496,0.427600,0.134432,1.455283
min,1.000000,0.000000,-1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-50.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,50.000000,76.666667,80.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,15.277778,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
50%,1.000000,0.000000,0.500000,0.000000,50.000000,80.000000,82.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,30.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,1.000000
75%,2.000000,1.000000,1.000000,0.000000,50.000000,90.000000,90.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,1.000000,0.000000,40.000000,0.

Saving final features dataset

In [88]:
df_cc_calls_agg.to_csv('../../../data/interim/cc_calls_features.csv', index=False)
print("Saved => data/interim/cc_calls_features.csv")

Saved => data/interim/cc_calls_features.csv


### Hypothesis

Hypothesis 1 => Customers with higher number of issues are more likely to churn

Hypothesis 2 => Customers with lower sentiment scores tend to churn

Hypothesis 3 => Customers who complain more frequently are more likely to churn

Hypothesis 4 => Customers with fewer interactions (calls) are more likely to churn

Hypothesis 5 => Negative experience over time (low sentiment + high issues) increases churn probability